In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.patches import Patch
import seaborn as sns
import sys
sys.path.append("/home/jg2447/slayman/motif_inference/MOBI/scripts/")
from src import plt_func
from src import motif_files

### data - performance

In [ ]:
def get_data_performance(performance_dir):
    df1 = pd.read_csv(performance_dir+"/DREME_top5_cv2.txt", sep="\t", header=None)
    df2 = pd.read_csv(performance_dir+"/HOMER_top5_cv2.txt", sep="\t", header=None)
    df3 = pd.read_csv(performance_dir+"/MEME_top5_cv2.txt", sep="\t", header=None)

    plt_data_mean_hit = (df1.mean()[0], df1.mean()[1], df2.mean()[0], df2.mean()[1], df3.mean()[0], df3.mean()[1])
    plt_data_sem_hit = (df1.sem()[0], df1.sem()[1], df2.sem()[0], df2.sem()[1], df3.sem()[0], df3.sem()[1])
    plt_data_mean_known = (df1.mean()[5], df1.mean()[6], df2.mean()[5], df2.mean()[6], df3.mean()[5], df3.mean()[6])
    plt_data_sem_known = (df1.sem()[5], df1.sem()[6], df2.sem()[5], df2.sem()[6], df3.sem()[5], df3.sem()[6])

    return((plt_data_mean_hit, plt_data_sem_hit, plt_data_mean_known, plt_data_sem_known))

In [ ]:
data1 = pd.DataFrame(get_data_performance("/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/fly/performance/"))
data2 = pd.DataFrame(get_data_performance("/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/worm/performance/"))
data3 = pd.DataFrame(get_data_performance("/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/humanGM12878/performance/"))
data4 = pd.DataFrame(get_data_performance("/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/humanK562/performance/"))

for data in (data1, data2, data3, data4):
    data.columns = ["DREME_baseline", "DREME_MOBI", "HOMER_baseline", "HOMER_MOBI", "MEME_baseline", "MEME_MOBI"]
    data.index = ["accuracy_mean", "accuracy_sem", "coverage_mean", "coverage_sem"]

data = pd.concat([data1.stack().reset_index(), data2.stack().reset_index(), data3.stack().reset_index(), data4.stack().reset_index()])
data["sample"] = ["fly"]*24+["worm"]*24+["GM12878"]*24+["K562"]*24
data.columns = ["measure", "tool_type", "value", "sample"]
data.to_csv("./data_performance.txt", sep="\t", index=False)

### data-entropy

In [ ]:
def get_n2a_known(mfile):
    motifs = []
    motif_names = []
    with open(mfile, "r") as f:
        lines = f.readlines()
        for k in range(len(lines)):
            if lines[k].startswith("letter-probability"):
                j = k
                while True:
                    j = j+1
                    if j < len(lines):
                        if lines[j].startswith(" 0.") or lines[j].startswith(" 1."):
                            pass
                        else:
                            break
                    else:
                        break
                motif = np.array(lines)[k+1:j]
                motif_array = []
                for i in motif:
                    motif_array.append(i.rstrip().split("\t"))
                motif = np.array(motif_array).astype(float)
                motifs.append(motif)
            if lines[k].startswith("MOTIF"):
                name = lines[k].split(" ")[1].rstrip()
                motif_names.append(name)
    n2a = {}
    for i,j in zip(motifs, motif_names):
        n2a[j] = i
    return(n2a)

In [ ]:
all_known_motifs = [i for i in os.listdir("/home/jg2447/slayman/data/motif/cisbp/meme_human_stack/") if i.endswith(".meme")]
entropy_list = []
entropy_list_core = []
for motif_fname in all_known_motifs:
    n2a = get_n2a_known("/home/jg2447/slayman/data/motif/cisbp/meme_human_stack/%s" % motif_fname)
    motif_entropy_list = []
    motif_core_entropy_list = []
    for motif_array in n2a.values():
        motif_entropy_list.append(motif_files.motif_entropy(motif_array))
        if motif_array.shape[0] > 8:
            index_start = motif_array.shape[0]// 2 - 4
            motif_array_core = motif_array[index_start:(index_start+8)]
        else:
            motif_array_core = motif_array
        motif_core_entropy_list.append(motif_files.motif_entropy(motif_array_core))
        
    entropy_list.append(np.mean(motif_entropy_list))
    entropy_list_core.append(np.mean(motif_core_entropy_list))
known = pd.DataFrame([[i.replace(".meme", "") for i in all_known_motifs], entropy_list, entropy_list_core]).T
known.columns = ["TF", "entropy", "entropy_core"]

In [ ]:
entropy_data = pd.DataFrame([])
for tt in ["DREME", "HOMER", "MEME"]:
    for cc in ["GM12878", "K562"]:
        spp = pd.read_csv("/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/human%s/tomtom_summary/%s_RankSPP_100_top5.txt" % (cc, tt), sep="\t").loc[:,["TF", "top_entropy"]]
        spp_short = pd.read_csv("/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/human%s/tomtom_summary/%s_RankSPP_20_top5.txt" % (cc, tt), sep="\t").loc[:,["TF", "top_entropy"]]
        mobi = pd.read_csv("/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/human%s/tomtom_summary/%s_RankLinear_0.9_20_top5.txt" % (cc, tt), sep="\t").loc[:,["TF", "top_entropy"]]
        entropy_df = pd.merge(spp, spp_short, left_on="TF", right_on="TF")
        entropy_df = pd.merge(entropy_df, mobi, left_on="TF", right_on="TF")
        entropy_df = pd.merge(known, entropy_df, left_on="TF", right_on="TF")
        entropy_df.columns = ["TF", "Known", "Known_core", "SPP", "SPP_short", "MOBI"]
        entropy_df = entropy_df.set_index("TF").unstack().reset_index()
        entropy_df.columns = ["Type", "TF", "Entropy"]
        entropy_df["Entropy"] = entropy_df["Entropy"].apply(pd.to_numeric)
        entropy_df["Type"] = entropy_df["Type"].astype('category')
        entropy_df["Cell"] = cc
        entropy_df["Tool"] = tt
        entropy_data = pd.concat([entropy_data, entropy_df])
entropy_data.to_csv("./data_entropy.tsv", sep="\t", index=False)

### data-inference

In [ ]:
inference_data = pd.DataFrame([])
for tt in ["DREME", "HOMER", "MEME"]:
    for cc in ["GM12878", "K562"]:
        spp = pd.read_csv("/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/human%s/tomtom_summary/%s_RankSPP_100_top5.txt" % (cc, tt), sep="\t").loc[:,["TF", "accuracy"]]
        spp_short = pd.read_csv("/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/human%s/tomtom_summary/%s_RankSPP_20_top5.txt" % (cc, tt), sep="\t").loc[:,["TF", "accuracy"]]
        mobi = pd.read_csv("/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/human%s/tomtom_summary/%s_RankLinear_0.9_20_top5.txt" % (cc, tt), sep="\t").loc[:,["TF", "accuracy"]]
        inference_df = pd.merge(spp, spp_short, left_on="TF", right_on="TF")
        inference_df = pd.merge(inference_df, mobi, left_on="TF", right_on="TF")
        inference_df.columns = ["TF", "SPP", "SPP_short", "MOBI"]
        inference_df = inference_df.set_index("TF").unstack().reset_index()
        inference_df.columns = ["Type", "TF", "Acc"]
        inference_df["Acc"] = inference_df["Acc"].apply(pd.to_numeric)
        inference_df["Type"] = inference_df["Type"].astype('category')
        inference_df["Cell"] = cc
        inference_df["Tool"] = tt
        inference_data = pd.concat([inference_data, inference_df])
inference_data.to_csv("./data_inference.tsv", sep="\t", index=False)

### test

In [ ]:
from scipy.stats import wilcoxon
data_entropy = pd.read_csv("./data_entropy.tsv", sep="\t").fillna(0)
data_inference = pd.read_csv("./data_inference.tsv", sep="\t").fillna(0)
data_performance = pd.read_csv("./data_performance.txt", sep="\t").fillna(0)

In [ ]:
wilcoxon(
    data_entropy.query('Tool == "HOMER" & Type == "Known_core"')['Entropy'].values,
    data_entropy.query('Tool == "HOMER" & Type == "SPP"')['Entropy'].values,
    alternative="greater")

In [ ]:
performance_dir = "/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/humanK562/performance/"
df1 = pd.read_csv(performance_dir+"/DREME_top5.txt", sep="\t", header=None)
df2 = pd.read_csv(performance_dir+"/MEME_top5.txt", sep="\t", header=None)
df3 = pd.read_csv(performance_dir+"/HOMER_top5.txt", sep="\t", header=None)

In [ ]:
(df1[1] >= df1[0]).sum() >= 3800, (df1[1] >= df1[0]).sum() >= 3960, (df1[1] >= df1[0]).sum() >= 3996 

In [ ]:
(df2[1] >= df2[0]).sum() >= 3800, (df2[1] >= df2[0]).sum() >= 3960, (df2[1] >= df2[0]).sum() >= 3996 

In [ ]:
(df3[1] >= df3[0]).sum() >= 3800, (df3[1] >= df3[0]).sum() >= 3960, (df3[1] >= df3[0]).sum() >= 3996 

In [ ]:
(df1[6] >= df1[5]).sum() >= 3800, (df1[6] >= df1[5]).sum() >= 3960, (df1[6] >= df1[5]).sum() >= 3996 

In [ ]:
(df2[6] >= df2[5]).sum() >= 3800, (df2[6] >= df2[5]).sum() >= 3960, (df2[6] >= df2[5]).sum() >= 3996 

In [ ]:
(df3[6] >= df3[5]).sum() >= 3800, (df3[6] >= df3[5]).sum() >= 3960, (df3[6] >= df3[5]).sum() >= 3996 

### plot

In [ ]:
def plt_bar(ax, data_mean, data_sem=None, yticks=None, color=["#ff5f69", "#bfd2ef"]):
    ax.bar([0,0.55, 3.5,4.05, 1.75,2.3], 
       data_mean,
       width=0.55,
       color=color,
       linewidth=0.3,
       edgecolor="black",
       align="edge")

    if np.any(data_sem):
        _, caps, sticks = ax.errorbar(
            x=np.array([0,0.55, 1.75,2.3, 3.5,4.05])+0.275, 
            y=data_mean,
            yerr=data_sem,
            fmt=".", lw=1, capsize=1, markersize=0, elinewidth=0.5, ecolor=(0.1, 0.1, 0.1, 0.75))
        for cap in caps:
             cap.set_markeredgewidth(0.5)

    ax.set_xlim([-0.5, 5.05])
    ax.xaxis.set_ticks([0.575, 2.325, 4.075]) # limit x range        
    ax.xaxis.set_ticklabels(["DREME", "MEME", "HOMER"], fontsize=x_tick_label_fs)
    ax.xaxis.set_tick_params(size=0, pad=1)

    # y axis
    ax.spines["left"].set_linewidth(0.5) # axis width
    ax.yaxis.set_ticks(yticks) # limit y range
    ax.yaxis.set_major_formatter(mpl.ticker.FormatStrFormatter('%.2f')) # use %.2f for number in yticklabels
    ax.yaxis.set_tick_params(size=2, width=0.6, pad=0, labelsize=y_tick_label_fs)
    

    # trim axis
    plt_func.ax_trim(ax)
    # no frame border
    ax.spines['bottom'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    return(ax)

In [ ]:
def plt_fig4a(ax_in, data):
    ax_in.annotate(
        "Correct inference ",
        xy=(-0.5, 0.35),
        rotation=90, xycoords="axes fraction", fontsize=y_label_fs) # large ylabel

    gs = mpl.gridspec.GridSpecFromSubplotSpec(
        9, 1, subplot_spec=ax_in, 
        hspace=0, height_ratios=[0.1, 0.1, 1, 0.28, 1, 0.28, 1, 0.28, 1])

    ## fly
    ax = fig.add_subplot(gs[2, 0])
    plt_bar(
        ax, 
        data.query('measure == "accuracy_mean" & sample == "fly"')['value'].values,
        #data.query('measure == "accuracy_sem" & sample == "fly"')['value'].values,
        yticks=[0, 0.6, 1.2], color=color)
    ax.set_ylabel("Fly", fontsize=y_label_fs, labelpad=2)

    ax.text(-0.4, 1.1, "A", transform=ax.transAxes, size=panel_number_fs, weight='bold')

    ## worm
    ax = fig.add_subplot(gs[4, 0])
    plt_bar(
        ax, 
        data.query('measure == "accuracy_mean" & sample == "worm"')['value'].values,
        #data.query('measure == "accuracy_sem" & sample == "worm"')['value'].values,
        yticks=[0, 0.6, 1.2], color=color)
    ax.set_ylabel("Worm", fontsize=y_label_fs, labelpad=2)

    ## GM12878
    ax = fig.add_subplot(gs[6, 0])
    plt_bar(
        ax, 
        data.query('measure == "accuracy_mean" & sample == "GM12878"')['value'].values,
        #data.query('measure == "accuracy_sem" & sample == "GM12878"')['value'].values,
        yticks=[0, 1.5, 3], color=color)
    ax.set_ylabel("GM12878", fontsize=y_label_fs, labelpad=2)

    ## K562
    ax = fig.add_subplot(gs[8, 0])
    plt_bar(
        ax, 
        data.query('measure == "accuracy_mean" & sample == "K562"')['value'].values,
        #data.query('measure == "accuracy_sem" & sample == "K562"')['value'].values,
        yticks=[0, 1.5, 3], color=color)
    ax.set_ylabel("K562", fontsize=y_label_fs, labelpad=2)

    ## Legend
    ax = plt.gca()
    patches = ax.patches
    ax_cbar = fig.add_subplot(gs[0, 0])
    ax_cbar.set_axis_off()
    ax_cbar.legend(
        patches, ["Conventional", "MOBI"], 
        loc="center", ncol=2, fontsize=legend_fs, frameon=False, handletextpad=0.5, columnspacing=1, handlelength=1.5)

In [ ]:
def plt_fig4b(ax_in, data):
    ax_in.annotate(
        "$\it{In}$ $\it{vitro}$ motif coverage", 
        xy=(-0.5, 0.28), 
        rotation=90, xycoords="axes fraction", fontsize=y_label_fs)

    gs = mpl.gridspec.GridSpecFromSubplotSpec(
        9, 1, subplot_spec=ax_in, 
        hspace=0, height_ratios=[0.1,0.1, 1,0.28, 1,0.28, 1,0.28, 1])

    #### fly
    ax = fig.add_subplot(gs[2, 0])
    plt_bar(
        ax, 
        data.query('measure == "coverage_mean" & sample == "fly"')['value'].values,
        #data.query('measure == "coverage_sem" & sample == "fly"')['value'].values,
        yticks=[0, 0.4, 0.8], color=color)
    ax.set_ylabel("Fly", fontsize=y_label_fs, labelpad=2)

    ax.text(-0.4, 1.1, "B", transform=ax.transAxes, size=panel_number_fs, weight='bold')

    #### worm
    ax = fig.add_subplot(gs[4, 0])
    plt_bar(
        ax, 
        data.query('measure == "coverage_mean" & sample == "worm"')['value'].values,
        #data.query('measure == "coverage_sem" & sample == "worm"')['value'].values,
        yticks=[0, 0.4, 0.8], color=color)
    ax.set_ylabel("Worm", fontsize=y_label_fs, labelpad=2)

    #### GM12878
    ax = fig.add_subplot(gs[6, 0])
    plt_bar(
        ax, 
        data.query('measure == "coverage_mean" & sample == "GM12878"')['value'].values,
        #data.query('measure == "coverage_sem" & sample == "GM12878"')['value'].values,
        yticks=[0, 0.75, 1.5], color=color)
    ax.set_ylabel("GM12878", fontsize=y_label_fs, labelpad=2)

    #### K562
    ax = fig.add_subplot(gs[8, 0])
    plt_bar(
        ax, 
        data.query('measure == "coverage_mean" & sample == "K562"')['value'].values,
        #data.query('measure == "coverage_sem" & sample == "K562"')['value'].values,
        yticks=[0, 0.75, 1.5], color=color)
    ax.set_ylabel("K562", fontsize=y_label_fs, labelpad=2)

    ## Legend
    ax = plt.gca()
    patches = ax.patches
    ax_cbar = fig.add_subplot(gs[0, 0])
    ax_cbar.set_axis_off()
    ax_cbar.legend(
        patches, ["Conventional", "MOBI"], 
        loc="center", ncol=2, fontsize=legend_fs, frameon=False, handletextpad=0.5, columnspacing=1, handlelength=1.5)

In [ ]:
def plt_fig4c(ax_in, data_entropy):
    flierprops = dict(marker='.', markersize=0.5, linestyle='none', markeredgecolor='black')
    boxprops = dict(linewidth=0.3)
    capprops = dict(linewidth=0.3)
    whiskerprops = dict(linewidth=0.3)
    
    box = ax_in.boxplot(
        np.array([
            data_entropy.query('Tool == "DREME" & Type == "SPP"')['Entropy'].values,
            data_entropy.query('Tool == "MEME" & Type == "SPP"')['Entropy'].values,
            data_entropy.query('Tool == "HOMER" & Type == "SPP"')['Entropy'].values]).T,
        positions=[1, 4, 7], patch_artist=True, showfliers=False, widths=0.5, flierprops=flierprops, boxprops=boxprops, capprops=capprops, whiskerprops=whiskerprops)

    plt.setp(box['medians'], color="black", linewidth=0.3)
    plt.setp(box["boxes"], facecolor="#ff5f69")

    box = ax_in.boxplot(
        np.array([
            data_entropy.query('Tool == "DREME" & Type == "SPP_short"')['Entropy'].values,
            data_entropy.query('Tool == "MEME" & Type == "SPP_short"')['Entropy'].values,
            data_entropy.query('Tool == "HOMER" & Type == "SPP_short"')['Entropy'].values]).T,
        positions=[1.75, 4.75, 7.75], patch_artist=True, showfliers=False, widths=0.5, flierprops=flierprops, boxprops=boxprops, capprops=capprops, whiskerprops=whiskerprops)

    plt.setp(box['medians'], color="black", linewidth=0.3)
    plt.setp(box["boxes"], facecolor="#ffffaa")

    box = ax_in.boxplot(
        np.array([
            data_entropy.query('Tool == "DREME" & Type == "MOBI"')['Entropy'].values,
            data_entropy.query('Tool == "MEME" & Type == "MOBI"')['Entropy'].values,
            data_entropy.query('Tool == "HOMER" & Type == "MOBI"')['Entropy'].values]).T,
        positions=[2.5, 5.5, 8.5], patch_artist=True, showfliers=False, widths=0.5, flierprops=flierprops, boxprops=boxprops, capprops=capprops, whiskerprops=whiskerprops)

    plt.setp(box['medians'], color="black", linewidth=0.3)
    plt.setp(box["boxes"], facecolor="#bfd2ef")

    box = ax_in.boxplot(
        data_entropy.query('Tool == "DREME" & Type == "Known_core"')['Entropy'].values,
        positions=[10], showfliers=False, widths=0.5, flierprops=flierprops, boxprops=boxprops, capprops=capprops, whiskerprops=whiskerprops)

    plt.setp(box['medians'], color="black", linewidth=0.3)
    
    ax_in.set_ylabel("Motif entropy", fontsize=y_label_fs, labelpad=1)
    ax_in.yaxis.set_tick_params(size=2, width=0.6)
    ax_in.yaxis.set_ticks([0, 0.7, 1.4])
    ax_in.yaxis.set_ticklabels([0, 0.7, 1.4],fontsize=y_tick_label_fs)

    ax_in.spines['right'].set_visible(False)
    ax_in.spines['top'].set_visible(False)

    ax_in.xaxis.set_ticks([0.75, 1.75, 4.75, 7.75, 10.3])
    ax_in.xaxis.set_ticklabels(["", "DREME", "MEME", "HOMER", "Known"], fontsize=x_tick_label_fs-0.5)
    ax_in.xaxis.set_tick_params(size=0, pad=0)

    ax_in.set_xlabel("")
    plt_func.ax_trim(ax_in)
    
    ax_in.spines['left'].set_linewidth(0.6)
    ax_in.spines['bottom'].set_visible(False)
    
    
    legend_elements = [
        Patch(facecolor='#ff5f69', edgecolor='black', linewidth=0.3, label="Strong binding sites \n(200bp)"),
        Patch(facecolor='#ffffaa', edgecolor='black', linewidth=0.3, label="Strong binding sites \n(40bp)"),
        Patch(facecolor='#bfd2ef', edgecolor='black', linewidth=0.3, label="MOBI \n(40bp)")]
    ax_in.legend(handles=legend_elements, loc="upper left", ncol=1, fontsize=legend_fs-1, frameon=False, handlelength=1.3, bbox_to_anchor=(0.02,1.05), handletextpad=0.5)
    
    ax_in.text(-0.4, 0.96, "C", transform=ax_in.transAxes, size=panel_number_fs, weight='bold')

In [ ]:
def plt_fig4d(ax_in, data_inference):
    box = ax_in.bar(
        [1, 4, 7],
        np.array([
            data_inference.query('Tool == "DREME" & Type == "SPP"')['Acc'].mean(),
            data_inference.query('Tool == "MEME" & Type == "SPP"')['Acc'].mean(),
            data_inference.query('Tool == "HOMER" & Type == "SPP"')['Acc'].mean()]).T,
        width=0.5,
        color="#ff5f69",
        linewidth=0.3,
        edgecolor="black",
        align="center")

    box = ax_in.bar(
        [1.75, 4.75, 7.75],
        np.array([
            data_inference.query('Tool == "DREME" & Type == "SPP_short"')['Acc'].mean(),
            data_inference.query('Tool == "MEME" & Type == "SPP_short"')['Acc'].mean(),
            data_inference.query('Tool == "HOMER" & Type == "SPP_short"')['Acc'].mean()]).T,
        width=0.5,
        color="#ffffaa",
        linewidth=0.3,
        edgecolor="black",
        align="center")

    box = ax_in.bar(
        [2.5, 5.5, 8.5],
        np.array([
            data_inference.query('Tool == "DREME" & Type == "MOBI"')['Acc'].mean(),
            data_inference.query('Tool == "MEME" & Type == "MOBI"')['Acc'].mean(),
            data_inference.query('Tool == "HOMER" & Type == "MOBI"')['Acc'].mean()]).T,
        width=0.5,
        color="#bfd2ef",
        linewidth=0.3,
        edgecolor="black",
        align="center")
    
    ax_in.set_ylabel("Correct inference", fontsize=y_label_fs, labelpad=1)
    ax_in.yaxis.set_tick_params(size=2, width=0.6)
    ax_in.yaxis.set_ticks([0, 1, 2, 3])
    ax_in.yaxis.set_ticklabels([0, 1, 2, 3],fontsize=y_tick_label_fs)

    ax_in.spines['right'].set_visible(False)
    ax_in.spines['top'].set_visible(False)

    ax_in.xaxis.set_ticks([1.75, 4.75, 7.75])
    ax_in.xaxis.set_ticklabels(["DREME", "MEME", "HOMER"], fontsize=x_tick_label_fs-0.5)
    ax_in.xaxis.set_tick_params(size=0, pad=0.5)

    ax_in.set_xlabel("")
    plt_func.ax_trim(ax_in)
    
    ax_in.spines['left'].set_linewidth(0.6)
    ax_in.spines['bottom'].set_visible(False)
    
    ax_in.text(-0.4, 0.96, "D", transform=ax_in.transAxes, size=panel_number_fs, weight='bold')

In [ ]:
figsize = (3.2, 4.48)
panel_number_fs = 8
x_tick_label_fs = 5
y_tick_label_fs = 5
x_label_fs = 7
y_label_fs = 7
title_fs = 7
legend_fs = 5

color = ["#ff5f69", "#bfd2ef"]

sns.set_context("paper")

fig, axs = plt.subplots(
    nrows=2, ncols=2,
    gridspec_kw={'hspace': 0.1, 'wspace': 0.6, 'height_ratios': [1, 0.4]},
    figsize=figsize, dpi=300)

axs[0, 0].axis('off') # A
axs[0, 1].axis('off') # B
#axs[1, 0].axis('off') # C
#axs[1, 1].axis('off') # D

# read data
data_performance = pd.read_csv("./data_performance.txt", sep="\t")
data_entropy = pd.read_csv("./data_entropy.tsv", sep="\t").fillna(0)
data_inference = pd.read_csv("./data_inference.tsv", sep="\t").fillna(0)


plt_fig4a(axs[0, 0], data_performance)
plt_fig4b(axs[0, 1], data_performance)
plt_fig4c(axs[1, 0], data_entropy)
plt_fig4d(axs[1, 1], data_inference)

plt.savefig("./fig4.pdf", dpi="figure", bbox_inches="tight")